# 02 – Spatial Analysis

Demonstrate spatial autocorrelation (Moran's I), hotspot detection (Gi*),
and IDW interpolation on synthetic soundscape data.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

from src.spatial_analysis.interpolation import idw_surface
from src.utils.logger import setup_logger

setup_logger(level='INFO')
print('Imports OK')

## 1. Synthetic GeoDataFrame

In [ ]:
rng = np.random.default_rng(42)
n = 30
lons = rng.uniform(72.0, 73.5, n)
lats = rng.uniform(34.5, 35.5, n)
aci = rng.uniform(200, 800, n)
rms_db = rng.uniform(-40, -5, n)

gdf = gpd.GeoDataFrame(
    {'aci': aci, 'rms_db': rms_db},
    geometry=[Point(lon, lat) for lon, lat in zip(lons, lats)],
    crs='EPSG:4326'
)
print(gdf.head())

## 2. Global Moran's I

In [ ]:
try:
    from src.spatial_analysis.spatial_stats import global_morans_i
    result = global_morans_i(gdf, column='aci', k=6)
    print(f"Moran's I  : {result['I']:.4f}")
    print(f"z-score    : {result['z_score']:.4f}")
    print(f"p-value    : {result['p_value']:.4f}")
except ImportError:
    print('esda / libpysal not installed – skipping Moran\'s I')

## 3. Hotspot Detection (Gi*)

In [ ]:
try:
    from src.spatial_analysis.hotspot_analysis import getis_ord_gi_star
    hotspots = getis_ord_gi_star(gdf, column='aci', k=6)
    print(hotspots['hotspot_type'].value_counts())
except ImportError:
    print('esda not installed – skipping Gi*')

## 4. IDW Interpolation

In [ ]:
grid_x, grid_y, z = idw_surface(gdf, column='aci', grid_resolution=0.05)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(grid_x, grid_y, z, cmap='YlOrRd', shading='auto')
ax.scatter(lons, lats, c=aci, cmap='YlOrRd', edgecolors='k', s=50, zorder=5)
plt.colorbar(im, ax=ax, label='ACI')
ax.set_title('IDW Interpolated ACI Surface')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()